In [78]:
import torch
from torch import nn
from d2l import torch as d2l
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim

In [79]:
# Load Data
df = pd.read_csv("../../Data/Final/Imputation/changi_imp_final.csv")


# Extract Year and Period from the Date column
df["Year"] = df["Date"].str.extract(r'(\d{4})').astype(int)
bimonthly_mapping = {'Jan-Feb': 0, 'Mar-Apr': 1, 'May-Jun': 2, 'Jul-Aug': 3, 'Sep-Oct': 4, 'Nov-Dec': 5}
df["Period"] = df["Date"].str[:6].map(bimonthly_mapping)

# Sort the data by (x, y, Year, Period)
df = df.sort_values(by=["x", "y", "Year", "Period"]).reset_index(drop=True)
df.drop(columns=["Period"], inplace=True)

# Check the first few rows to verify the sorting
print(df.head())
print(len(df))


            x         y          Date      Value  Year
0  103.964566  1.350459  Mar-Apr 2000  23.775064  2000
1  103.964566  1.350459  May-Jun 2000  27.537897  2000
2  103.964566  1.350459  Jul-Aug 2000  27.519318  2000
3  103.964566  1.350459  Sep-Oct 2000  32.314507  2000
4  103.964566  1.350459  Nov-Dec 2000  23.391225  2000
788210


In [101]:
import pandas as pd

def map_date_to_index(df):
    # Mapping months (this could be expanded or adjusted)
    month_map = {'Jan-Feb': 1, 'Mar-Apr': 3, 'May-Jun': 5, 'Jul-Aug': 7, 'Sep-Oct': 9, 'Nov-Dec': 11}
    
    # Extract the month-year from the 'Date' column
    df['numeric_date'] = df['Date'].apply(lambda x: month_map[x.split()[0]] + (int(x.split()[1]) - 2000) * 12)
    
    return df

df = map_date_to_index(df)  # Add a numeric date column for processing
print(df.head())


            x         y          Date      Value  Year  numeric_date
0  103.964566  1.350459  Mar-Apr 2000  23.775064  2000             3
1  103.964566  1.350459  May-Jun 2000  27.537897  2000             5
2  103.964566  1.350459  Jul-Aug 2000  27.519318  2000             7
3  103.964566  1.350459  Sep-Oct 2000  32.314507  2000             9
4  103.964566  1.350459  Nov-Dec 2000  23.391225  2000            11


In [102]:
import numpy as np

def create_rolling_windows(df, window_size):
    windows = []
    max_date_index = df['numeric_date'].max()
    
    # For each rolling window, create training and validation sets
    for start_index in range(0, max_date_index - (window_size + 1) * 2, 2):  # Move by 2 months
        train_end_index = start_index + window_size * 2 - 1
        validation_start_index = train_end_index + 1
        validation_end_index = validation_start_index + 12 - 1
        
        # Extract train and validation data
        train_data = df[(df['numeric_date'] >= df['numeric_date'].iloc[start_index]) & 
                        (df['numeric_date'] <= df['numeric_date'].iloc[train_end_index])]
        
        validation_data = df[(df['numeric_date'] >= df['numeric_date'].iloc[validation_start_index]) & 
                             (df['numeric_date'] <= df['numeric_date'].iloc[validation_end_index])]
        
        windows.append((train_data, validation_data))
    
    return windows


In [103]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_layer_size=64, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_layer_size, batch_first=True)
        self.fc = nn.Linear(hidden_layer_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Using the output of the last time step
        return out


In [104]:
def train_lstm(train_loader, model, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        for data in train_loader:
            inputs = data[:, :-1, :]  # All but the last time step
            targets = data[:, -1, :]  # The last time step (forecast)
            
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()


In [105]:
from sklearn.metrics import mean_squared_error

def calculate_rmse(predictions, true_values):
    rmse_all = np.sqrt(mean_squared_error(true_values, predictions))
    
    # RMSE for specific forecasts (f_1, f_3, f_6, f_9, f_12)
    rmse_specific = [np.sqrt(mean_squared_error(true_values[i:i+1], predictions[i:i+1])) 
                     for i in [0, 2, 5, 8, 11]]
    
    return rmse_all, rmse_specific


In [113]:
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

# Define the LSTM model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_layer_size, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_layer_size, batch_first=True)
        self.fc = nn.Linear(hidden_layer_size, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        output = self.fc(lstm_out[:, -1, :])  # Get the output from the last time step
        return output

# Run the rolling window experiment
def run_rolling_window_experiment(df, window_sizes=[9, 10, 11, 12, 13], num_samples=10):
    results = []

    for window_size in window_sizes:
        # Get the rolling windows
        windows = create_rolling_windows(df, window_size)
        
        # Randomly pick 10 samples
        random_windows = random.sample(windows, num_samples)
        
        # Initialize lists for RMSE results
        rmse_all_list = []
        rmse_specific_list = []
        
        for train_data, validation_data in random_windows:
            # Z-score normalization (StandardScaler)
            scaler = StandardScaler()
            train_scaled = scaler.fit_transform(train_data[['Value']])
            validation_scaled = scaler.transform(validation_data[['Value']])

            # Prepare tensors for LSTM
            train_tensor = torch.tensor(train_scaled, dtype=torch.float32).unsqueeze(-1)  # Adding feature dimension
            validation_tensor = torch.tensor(validation_scaled, dtype=torch.float32).unsqueeze(-1)
            
            # Create DataLoader for batching
            train_loader = torch.utils.data.DataLoader(train_tensor, batch_size=32, shuffle=True)
            validation_loader = torch.utils.data.DataLoader(validation_tensor, batch_size=32, shuffle=False)

            # Initialize the model for each experiment
            model = LSTMModel(input_size=1, hidden_layer_size=64, output_size=1)
            criterion = nn.MSELoss()
            optimizer = optim.Adam(model.parameters(), lr=0.001)

            # Train the model
            train_lstm(train_loader, model, criterion, optimizer, epochs=5)

            # Make predictions on validation data
            predictions = []
            true_values = []
            model.eval()  # Set model to evaluation mode
            with torch.no_grad():
                for data in validation_loader:
                    inputs = data
                    predictions.append(model(inputs).numpy())  # Model output (predictions)
                    true_values.append(data.numpy())  # Actual values (truth)

            predictions = np.concatenate(predictions, axis=0)
            true_values = np.concatenate(true_values, axis=0)

            # Calculate RMSE
            rmse_all, rmse_specific = calculate_rmse(predictions, true_values)
            rmse_all_list.append(rmse_all)
            rmse_specific_list.append(rmse_specific)

        # Average RMSE statistics for each window size
        results.append({
            'window_size': window_size,
            'rmse_all_mean': np.mean(rmse_all_list),
            'rmse_specific_mean': np.mean(rmse_specific_list, axis=0)
        })

    return results

# Example call to run the experiment (use your DataFrame `df`)
# results = run_rolling_window_experiment(df)
# print(results)


In [114]:
results = run_rolling_window_experiment(df)
print(results)


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler.

In [115]:
import numpy as np
from sklearn.model_selection import train_test_split

# Group by coordinate and sort by date
grouped = df.groupby(["x", "y"])
all_sequences = []
all_targets = []

for (x, y), group in grouped:
    group = group.sort_values("Date")
    values = group["Value"].values
    
    # Create sliding windows and targets
    window_size = 54  # 9 years (9 * 6 bimonthly periods)
    horizon = 2       # Predict 2 bimonthly periods ahead
    
    # Use NumPy for efficient window generation
    windows = np.lib.stride_tricks.sliding_window_view(values, window_size)
    targets = np.column_stack([values[i + window_size : i + window_size + horizon] 
                              for i in range(len(values) - window_size - horizon + 1)])
    
    all_sequences.extend(windows)
    all_targets.extend(targets)

# Convert to 3D array for LSTM input (samples, time_steps, features)
X = np.array(all_sequences).reshape(-1, window_size, 1)  # Shape: (n_samples, 54, 1)
y = np.array(all_targets)                                # Shape: (n_samples, 2)


In [121]:
print(X.shape)
print(y.shape)

(507840, 54, 1)
(10580, 94)


In [127]:
import numpy as np
import pandas as pd

# Ensure proper sorting by converting Date to a structured format
def parse_date(date_str):
    """Convert 'Jan-Feb 2020' format to a sortable format."""
    months = {"Jan-Feb": 1, "Mar-Apr": 2, "May-Jun": 3, "Jul-Aug": 4, "Sep-Oct": 5, "Nov-Dec": 6}
    period, year = date_str.split()
    return int(year) * 10 + months[period]  # Creates a sortable integer

df["Date_Sortable"] = df["Date"].apply(parse_date)

# Group by coordinate and sort by Date
grouped = df.groupby(["x", "y"])
all_sequences = []
all_targets = []

window_size = 54  # 9 years (9 * 6 bimonthly periods)
horizon = 2       # Predict 2 bimonthly periods ahead

for (x, y), group in grouped:
    group = group.sort_values("Date_Sortable")  # Sort by the new sortable date format
    values = group["Value"].values

    if len(values) < window_size + horizon:
        continue  # Skip if there's not enough data

    # Create sliding windows and targets
    windows = np.lib.stride_tricks.sliding_window_view(values, window_size)
    targets = np.array([values[i + window_size : i + window_size + horizon] 
                    for i in range(len(values) - window_size - horizon + 1)])

    all_sequences.extend(windows)
    all_targets.extend(targets)

# Convert to NumPy arrays for LSTM input
X = np.array(all_sequences).reshape(-1, window_size, 1)  # Shape: (n_samples, 54, 1)
y = np.array(all_targets)  # Shape: (n_samples, 2)

print(f"Final dataset shape: X={X.shape}, y={y.shape}")


Final dataset shape: X=(507840, 54, 1), y=(497260, 2)
